Define the result folder

In [1]:
import yaml
from pathlib import Path
from ipyfilechooser import FileChooser
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import tifffile
from PIL import Image

srcdir, dstdir = "", ""
if Path("config.yml").exists():
    with open("config.yml", "r") as file:
        config = yaml.safe_load(file)
        if "source" in config.keys():
            srcdir = Path(config["source"])
        if "destination" in config.keys():
            dstdir = Path(config["destination"])

fc = FileChooser(dstdir, select_desc="Destination")
display(fc)

FileChooser(path='C:\Users\Amy Courtney\Documents\Temp\Marie_HCR_Test\Full_Size', filename='', title='', show_…

In [3]:
import pandas as pd

dstdir = Path(fc.selected) if fc.selected is not None else Path(dstdir)
filelistname = dstdir / "filelist.csv"
filelist = pd.read_csv(filelistname)
filelist

,folder,name,channel1,channel2,channel3,channel4
0,C:\Users\Amy Courtney\Documents\Temp\Marie_HCR...,3-01-Stitching-26_AllChannels_Crop1.ims,vacht,vglut,Cell_Label,nuclei
1,C:\Users\Amy Courtney\Documents\Temp\Marie_HCR...,4-01-Stitching-06_AllChannels_Crop1.ims,vacht,vglut,Cell_Label,nuclei


In [4]:
from itertools import chain, combinations
from functools import partial
from functools import reduce
import operator
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import napari
import dask


def get_files(dstdir, row, key=None):
    if key == "ims":
        return Path(row["folder"]) / row["name"]
    elif key == "regions":
        return Path(dstdir / str(row["name"]).replace(".ims", "-regions.json"))
    elif key == "labels":
        return Path(dstdir / str(row["name"]).replace(".ims", "-labels.tif"))
    elif key == "measurements":
        return Path(dstdir / str(row["name"]).replace(".ims", "-measurements.csv"))
    elif key == "stats":
        return Path(dstdir / str(row["name"]).replace(".ims", "-stats.csv"))
    else:
        return {
            "ims": get_files(dstdir, row, "ims"),
            "regions": get_files(dstdir, row, "regions"),
            "labels": get_files(dstdir, row, "labels"),
            "measurements": get_files(dstdir, row, "measurements"),
        }



In [5]:
import ipywidgets as widgets

w = widgets.Dropdown(
    options=[(x, k) for k, x in enumerate(filelist["name"])],
    value=1,
    description="Image:",
)
display(w)

Dropdown(description='Image:', index=1, options=(('3-01-Stitching-26_AllChannels_Crop1.ims', 0), ('4-01-Stitch…

In [7]:
#updated
from imaris_ims_file_reader.ims import ims
import numpy as np
import napari
import tifffile
import os
import pandas as pd

# --- Load IMS image ---
row = filelist.iloc[w.value]
resolution_level = 1

img_path = get_files(dstdir, row, "ims")
image_name = os.path.splitext(os.path.basename(img_path))[0]

img = ims(
    img_path,
    ResolutionLevelLock=resolution_level,
    squeeze_output=False,
)

# data shape: (1, 4, Z, Y, X)
data = img[:]
data = np.squeeze(data, axis=0)    # (4, Z, Y, X)

# ---------------------------
# Only load channel 3
# ---------------------------
channel_index = 2    # 0=ch0, 1=ch1, 2=ch2, **3=ch3**
channel3 = data[channel_index]  # (Z, Y, X)

# --- Find the labels file in the same folder ---
folder = os.path.dirname(img_path)
labels_file = None

for f in os.listdir(folder):
    if f.startswith(image_name) and f.endswith("-labels.tif"):
        labels_file = os.path.join(folder, f)
        break

if labels_file is None:
    raise FileNotFoundError(
        f"No {image_name}*-labels.tif file found in folder: {folder}"
    )

# --- Load labels image (typically ZYX or 3D integer mask) ---
labels = tifffile.imread(labels_file)

# --- Create napari viewer and add ONLY channel 3 + labels ---
viewer = napari.Viewer()

viewer.add_image(
    channel3,       # (Z, Y, X)
    name="ch3",
    colormap="magenta",
)

labels_layer = viewer.add_labels(labels, name="Labels")

# --- In-memory storage of selected labels ---
saved_labels = set()
saved_label_info = []


def record_current_label():
    """Get label under cursor, compute details, and store in memory."""
    pos = np.round(viewer.cursor.position).astype(int)

    # Only take the last 3 entries → (z,y,x)
    zyx = tuple(pos[-labels.ndim:])

    # Identify label under cursor
    try:
        label_id = int(labels[zyx])
    except IndexError:
        print("Cursor is outside label volume.")
        return

    if label_id == 0:
        print("Background label 0 → not saving.")
        return

    if label_id in saved_labels:
        print(f"Label {label_id} already saved.")
        return

    coords = np.argwhere(labels == label_id)

    if coords.size == 0:
        print(f"No voxels found for label {label_id}.")
        return

    voxel_count = coords.shape[0]
    centroid = coords.mean(axis=0)  # (z,y,x)
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)

    info = {
        "label_id": int(label_id),
        "voxel_count": int(voxel_count),
        "centroid_z": float(centroid[0]),
        "centroid_y": float(centroid[1]),
        "centroid_x": float(centroid[2]),
        "bbox_min_z": int(mins[0]),
        "bbox_min_y": int(mins[1]),
        "bbox_min_x": int(mins[2]),
        "bbox_max_z": int(maxs[0]),
        "bbox_max_y": int(maxs[1]),
        "bbox_max_x": int(maxs[2]),
    }

    saved_label_info.append(info)
    saved_labels.add(label_id)
    print(f"Saved label {label_id}")
    print(info)


# Press "a" to save label under cursor
@viewer.bind_key("a")
def save_selected_label(v):
    record_current_label()


# --- Start napari ---
napari.run()


# ---------------------------
# After napari closes:
# Update measurements CSV
# ---------------------------

selected_ids = {info["label_id"] for info in saved_label_info}
print("Selected label IDs:", selected_ids)

if selected_ids:
    # find the measurements file
    measurements_file = None
    for f in os.listdir(folder):
        if f.startswith(image_name) and f.endswith("-measurements.csv"):
            measurements_file = os.path.join(folder, f)
            break

    if measurements_file is None:
        raise FileNotFoundError(
            f"No {image_name}*-measurements.csv found in folder: {folder}"
        )

    print("Updating:", measurements_file)

    m = pd.read_csv(measurements_file)

    # auto-detect label column
    label_col_candidates = ["label", "Label", "Label_ID", "LabelID"]
    label_col = next((c for c in label_col_candidates if c in m.columns), None)

    if label_col is None:
        raise ValueError(
            f"No label column found in measurements file. "
            f"Tried: {label_col_candidates}"
        )

    m[label_col] = m[label_col].astype(int)

    # Apply selection → Cell_Label = 1 or 0
    m["Cell_Label"] = m[label_col].isin(selected_ids).astype(int)

    m.to_csv(measurements_file, index=False)
    print("Done. Saved updated measurements file.")

else:
    print("No labels selected → measurements file not modified.")


Opening readonly file: C:\Users\Amy Courtney\Documents\Temp\Marie_HCR_Test\Cropped\3-01-Stitching-26_AllChannels_Crop1.ims 

Closing file: C:\Users\Amy Courtney\Documents\Temp\Marie_HCR_Test\Cropped\3-01-Stitching-26_AllChannels_Crop1.ims 

Selected label IDs: set()
No labels selected → measurements file not modified.


Saved label 34
{'label_id': 34, 'voxel_count': 2512, 'centroid_z': 2.9701433121019107, 'centroid_y': 116.47611464968153, 'centroid_x': 35.00915605095541, 'bbox_min_z': 1, 'bbox_min_y': 105, 'bbox_min_x': 17, 'bbox_max_z': 5, 'bbox_max_y': 129, 'bbox_max_x': 51}
Saved label 17
{'label_id': 17, 'voxel_count': 3808, 'centroid_z': 2.0438550420168067, 'centroid_y': 171.22767857142858, 'centroid_x': 56.46481092436975, 'bbox_min_z': 0, 'bbox_min_y': 150, 'bbox_min_x': 39, 'bbox_max_z': 5, 'bbox_max_y': 193, 'bbox_max_x': 77}
Background label 0 → not saving.


In [25]:
#new
import pandas as pd
import os

# --- Paths in same folder ---
measurements_file = os.path.join(folder, f"{image_name}-measurements.csv")

# --- Extract selected label IDs from in-memory variable ---
selected_ids = {entry["label_id"] for entry in saved_label_info}

print(f"Selected labels: {sorted(selected_ids)}")

# --- Load measurements ---
m = pd.read_csv(measurements_file)

# Make sure the label column is integer
m["label"] = m["label"].astype(int)

# Create/overwrite Cell_Label column
m["Cell_Label"] = m["label"].apply(lambda x: 1 if x in selected_ids else 0)

# --- Save back to the same measurements file ---
m.to_csv(measurements_file, index=False)
print(f"Updated measurements saved to: {measurements_file}")


Selected labels: [19]
Updated measurements saved to: C:\Users\Amy Courtney\Documents\Temp\Marie_HCR_Test\Cropped\4-01-Stitching-06_AllChannels_Crop1-measurements.csv
